# 🏦 Loan Default Prediction Model
**Author:** Mayowa Alamutu  
**Domain:** Fintech / Credit Risk  
**Tools:** Python, Pandas, Scikit-learn, XGBoost, Seaborn, SMOTE  

---

## Project Overview
In lending, a missed default costs far more than a rejected application.  
This notebook builds a binary classification model to predict loan default risk from borrower features — giving lenders an early-warning signal before disbursement.

**Goal:** Predict whether a borrower will default (1) or not (0)  
**Metric priority:** Recall on defaults (catching as many real defaults as possible)


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'monospace'
print("✅ Libraries loaded successfully")

## 2. Generate Synthetic Dataset
> *In a real project, replace this with your actual loan CSV. The structure mirrors Lending Club / fintech loan data.*

In [ ]:
np.random.seed(42)
n = 5000

# Borrower features
loan_amount   = np.random.randint(1000, 40000, n)
annual_income = np.random.randint(20000, 200000, n)
dti           = np.round(np.random.uniform(5, 50, n), 2)          # debt-to-income ratio
interest_rate = np.round(np.random.uniform(5.5, 28.0, n), 2)
emp_length    = np.random.choice([0,1,2,3,5,7,10], n)
loan_grade    = np.random.choice(['A','B','C','D','E','F','G'], n,
                                  p=[0.20,0.22,0.20,0.15,0.10,0.08,0.05])
home_ownership = np.random.choice(['RENT','OWN','MORTGAGE'], n, p=[0.45,0.20,0.35])
purpose       = np.random.choice(['debt_consolidation','credit_card',
                                   'home_improvement','other'], n,
                                  p=[0.40,0.25,0.20,0.15])

# Grade → default probability mapping (realistic)
grade_default_prob = {'A':0.04,'B':0.08,'C':0.15,'D':0.22,'E':0.32,'F':0.42,'G':0.52}
base_prob = np.array([grade_default_prob[g] for g in loan_grade])

# Adjust probability by DTI and interest rate
adj_prob = np.clip(base_prob + (dti - 20)/200 + (interest_rate - 15)/100, 0.02, 0.85)
default   = (np.random.uniform(0, 1, n) < adj_prob).astype(int)

df = pd.DataFrame({
    'loan_amount': loan_amount,
    'annual_income': annual_income,
    'dti': dti,
    'interest_rate': interest_rate,
    'emp_length': emp_length,
    'loan_grade': loan_grade,
    'home_ownership': home_ownership,
    'purpose': purpose,
    'default': default
})

print(f"Dataset shape: {df.shape}")
print(f"\nDefault rate: {df['default'].mean():.1%}")
print(f"\nClass distribution:\n{df['default'].value_counts()}")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Loan Default — Exploratory Data Analysis', fontsize=15, fontweight='bold', y=1.01)

# 1. Default rate by grade
grade_order = ['A','B','C','D','E','F','G']
grade_default = df.groupby('loan_grade')['default'].mean().reindex(grade_order) * 100
axes[0,0].bar(grade_order, grade_default.values, color=sns.color_palette('RdYlGn_r', 7))
axes[0,0].set_title('Default Rate by Loan Grade')
axes[0,0].set_ylabel('Default Rate (%)')
axes[0,0].yaxis.set_major_formatter(mtick.PercentFormatter())

# 2. DTI distribution by default
for label, grp in df.groupby('default'):
    axes[0,1].hist(grp['dti'], bins=30, alpha=0.6,
                   label='Default' if label==1 else 'No Default', density=True)
axes[0,1].set_title('DTI Distribution by Default Status')
axes[0,1].set_xlabel('Debt-to-Income Ratio (%)')
axes[0,1].legend()

# 3. Interest rate by default
df.boxplot(column='interest_rate', by='default', ax=axes[0,2],
           boxprops=dict(color='steelblue'), medianprops=dict(color='red'))
axes[0,2].set_title('Interest Rate by Default Status')
axes[0,2].set_xlabel('Default (0=No, 1=Yes)')
axes[0,2].set_ylabel('Interest Rate (%)')
plt.sca(axes[0,2])
plt.title('Interest Rate by Default Status')

# 4. Loan amount vs default
df.boxplot(column='loan_amount', by='default', ax=axes[1,0],
           boxprops=dict(color='steelblue'), medianprops=dict(color='red'))
axes[1,0].set_title('Loan Amount by Default Status')
axes[1,0].set_xlabel('Default (0=No, 1=Yes)')
plt.sca(axes[1,0])
plt.title('Loan Amount by Default Status')

# 5. Default by home ownership
home_default = df.groupby('home_ownership')['default'].mean() * 100
axes[1,1].bar(home_default.index, home_default.values, color=['#2196F3','#FF5722','#4CAF50'])
axes[1,1].set_title('Default Rate by Home Ownership')
axes[1,1].set_ylabel('Default Rate (%)')
axes[1,1].yaxis.set_major_formatter(mtick.PercentFormatter())

# 6. Correlation heatmap
num_cols = ['loan_amount','annual_income','dti','interest_rate','emp_length','default']
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1,2], square=True, cbar_kws={'shrink':0.8})
axes[1,2].set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ EDA complete")

## 4. Feature Engineering & Preprocessing

In [ ]:
# Feature engineering
df['loan_to_income']  = df['loan_amount'] / df['annual_income']
df['monthly_payment'] = (df['loan_amount'] * (df['interest_rate']/1200)) /                          (1 - (1 + df['interest_rate']/1200)**-36)

# Encode categoricals
le = LabelEncoder()
df['grade_encoded']     = le.fit_transform(df['loan_grade'])    # A=0 … G=6
df['ownership_encoded'] = le.fit_transform(df['home_ownership'])
df['purpose_encoded']   = le.fit_transform(df['purpose'])

feature_cols = ['loan_amount','annual_income','dti','interest_rate',
                'emp_length','grade_encoded','ownership_encoded',
                'purpose_encoded','loan_to_income','monthly_payment']

X = df[feature_cols]
y = df['default']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Scale for logistic regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)

print(f"Train size (after SMOTE): {X_train_sm.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\nClass balance after SMOTE:\n{pd.Series(y_train_sm).value_counts()}")

## 5. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05,
                                         max_depth=5, random_state=42,
                                         eval_metric='logloss', verbosity=0)
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_sc, y_train_sm)
        y_pred  = model.predict(X_test_sc)
        y_proba = model.predict_proba(X_test_sc)[:,1]
    else:
        model.fit(X_train_sm, y_train_sm)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:,1]

    auc = roc_auc_score(y_test, y_proba)
    results[name] = {'model': model, 'y_pred': y_pred,
                     'y_proba': y_proba, 'auc': auc}
    print(f"{'─'*50}")
    print(f"  {name}  |  ROC-AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['No Default','Default']))

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\n🏆 Best model: {best_name}  (AUC = {results[best_name]['auc']:.4f})")

## 6. Visualise Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Evaluation — Loan Default Prediction', fontsize=14, fontweight='bold')

# -- ROC Curves --
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={res['auc']:.2f})")
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_title('ROC Curves — All Models')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)

# -- Confusion Matrix (XGBoost) --
best = results['XGBoost']
cm = confusion_matrix(y_test, best['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['No Default','Default']).plot(
    ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Confusion Matrix — XGBoost')

# -- Feature Importance --
xgb_model = results['XGBoost']['model']
feat_imp = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values()
feat_imp.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Feature Importance — XGBoost')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Visualisations saved")

## 7. Key Findings & Business Interpretation

In [ ]:
best = results['XGBoost']
cm = confusion_matrix(y_test, best['y_pred'])
tn, fp, fn, tp = cm.ravel()

precision  = tp / (tp + fp)
recall     = tp / (tp + fn)
f1         = 2 * precision * recall / (precision + recall)

print("=" * 55)
print("  LOAN DEFAULT PREDICTION — SUMMARY RESULTS")
print("=" * 55)
print(f"  Best Model    : XGBoost")
print(f"  ROC-AUC Score : {best['auc']:.4f}")
print(f"  Accuracy      : {(tp+tn)/(tp+tn+fp+fn):.1%}")
print(f"  Precision     : {precision:.1%}  (of flagged, how many truly default)")
print(f"  Recall        : {recall:.1%}  (of all defaults, how many were caught)")
print(f"  F1 Score      : {f1:.4f}")
print()
print("  CONFUSION MATRIX:")
print(f"    True Negatives  (correctly safe)  : {tn}")
print(f"    True Positives  (correctly flagged): {tp}")
print(f"    False Positives (wrongly flagged)  : {fp}")
print(f"    False Negatives (missed defaults)  : {fn}")
print()
print("  BUSINESS IMPACT:")
print(f"    Per 1,000 applications screened,")
print(f"    the model prevents ~{int(tp/len(y_test)*1000)} defaults")
print(f"    while incorrectly rejecting ~{int(fp/len(y_test)*1000)} safe borrowers.")
print("=" * 55)

## 8. Next Steps
- Deploy model as a REST API endpoint (Flask / FastAPI)  
- Integrate real-time borrower data from loan management system  
- Add SHAP values for per-application explainability  
- Monitor for data drift quarterly — retrain as needed  

---
*Notebook by Mayowa Alamutu | [Portfolio](https://mayowahabeeb.framer.website/) | [LinkedIn](https://www.linkedin.com/in/mayowa-alamutu-84185a25a)*
